In [ ]:
%pip install cecil

In [ ]:
import json
import os

import boto3
import cecil
import wkls
from cecil.models.subscription import SubscriptionTIFF
from pyspark.sql import functions as F
from sedona.spark import SedonaContext

config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

### Get AOI (Benton County, Oregon)

In [ ]:
# Get Benton County OR boundary in GeoJSON and WKT formats
benton_geojson_str = wkls['us']['or']['Benton County'].geojson()
benton_wkt = wkls['us']['or']['Benton County'].wkt()

# Parse GeoJSON string to dict (needed for Cecil API)
benton_geojson = json.loads(benton_geojson_str)

print(f"Geometry type: {benton_geojson['type']}")
print(f"WKT preview: {benton_wkt[:120]}...")

### Pull data from Cecil

In [ ]:
import getpass
os.environ["CECIL_API_KEY"] = getpass.getpass("Cecil API Key: ")

import os
import boto3

s3_env_path = os.getenv("USER_S3_PATH") + "cecil/api.key"

bucket = s3_env_path.split("/")[2]
key = "/".join(s3_env_path.split("/")[3:])

s3 = boto3.client("s3")
obj = s3.get_object(Bucket=bucket, Key=key)
os.environ["CECIL_API_KEY"] = obj["Body"].read().decode("utf-8").strip()

In [ ]:
cecil_client = cecil.Client()

# Create an AOI using the WKLS geometry
aoi = cecil_client.create_aoi(
    external_ref="Benton County, OR",
    geometry=benton_geojson,
)

# UMD Hansen Global Forest Change ID on Cecil
HANSEN_ID = "9659ec1d-7091-4f8b-9db5-e9fe07d2f508"

subscription = cecil_client.create_subscription(
    external_ref="Hansen - Benton County OR",
    aoi_id=aoi.id,
    dataset_id=HANSEN_ID,
)

print(f"AOI ID: {aoi.id}")
print(f"Area: {aoi.hectares:.0f} hectares")
print(f"Subscription ID: {subscription.id}")

### Get S3 Paths

In [ ]:
subscription_id = subscription.id  # use the subscription we just created

# Call Cecil API to get S3 credentials and file metadata
res = SubscriptionTIFF(
    **cecil_client._get(url=f"/v0/subscriptions/{subscription_id}/files/tiff")
)

print(f"Dataset:  {res.dataset_name}")
print(f"Bucket:   {res.bucket.name}")
print(f"Prefix:   {res.bucket.prefix}")
print(f"Region:   {res.credentials.region}")
print(f"Expires:  {res.credentials.expiration}")
print(f"\nBand metadata:")
for filename, file_info in res.file_mapping.items():
    for band in file_info.bands:
        print(f"  {filename}: band {band.number} = '{band.name}' ({band.dtype}, nodata={band.nodata})")

In [ ]:
# List GeoTIFF files under the subscription prefix
session = boto3.session.Session(
    aws_access_key_id=res.credentials.access_key_id,
    aws_secret_access_key=res.credentials.secret_access_key,
    aws_session_token=res.credentials.session_token,
    region_name=res.credentials.region,
)

s3 = session.client("s3")
paginator = s3.get_paginator("list_objects_v2")

tiff_keys = []
for page in paginator.paginate(Bucket=res.bucket.name, Prefix=res.bucket.prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].lower().endswith((".tif", ".tiff")):
            tiff_keys.append(obj["Key"])

print(f"Found {len(tiff_keys)} GeoTIFF file(s):")
for key in tiff_keys:
    print(f"  s3a://{res.bucket.name}/{key}")

In [ ]:
# Build the Hadoop S3A credential string for RS_FromPath
cred_params = (
    f"fs.s3a.access.key={res.credentials.access_key_id};"
    f"fs.s3a.secret.key={res.credentials.secret_access_key};"
    f"fs.s3a.session.token={res.credentials.session_token}"
)

# Build S3A paths (note: s3a:// protocol, not s3://)
s3a_paths = [f"s3a://{res.bucket.name}/{key}" for key in tiff_keys]

# Create a DataFrame of S3 paths
paths_df = sedona.createDataFrame([(p,) for p in s3a_paths], ["path"])

# Load as out-db rasters with Cecil credentials
raster_df = paths_df.withColumn(
    "rast",
    F.expr(f"RS_FromPath(path, '{cred_params}')")
)

raster_df.cache()
raster_df.createOrReplaceTempView("cdl_rasters")

print(f"Loaded {raster_df.count()} CDL raster(s)")
raster_df.selectExpr(
    "path",
    "RS_Width(rast) AS width",
    "RS_Height(rast) AS height",
    "RS_SRID(rast) AS srid",
    "RS_NumBands(rast) AS num_bands",
).show(truncate=60)

### Overall summary stats for Benton County

In [ ]:
# Register each band as its own view
for variable in ["data_mask", "forest_gain", "loss_year", "tree_cover"]:
    raster_df.filter(F.col("path").contains(variable)) \
        .createOrReplaceTempView(variable)

# Zonal stats per band over the AOI polygon
hansen_stats = sedona.sql("""
    SELECT
        ST_GeomFromWKT('""" + benton_wkt + """') AS geometry,
        RS_ZonalStats(ly.rast, ST_GeomFromWKT('""" + benton_wkt + """'), 1, 'count', true) AS total_pixels,
        RS_ZonalStats(tc.rast, ST_GeomFromWKT('""" + benton_wkt + """'), 1, 'mean', true)  AS mean_tree_cover_pct,
        RS_ZonalStats(fg.rast, ST_GeomFromWKT('""" + benton_wkt + """'), 1, 'sum', true)   AS gain_pixels,
        RS_ZonalStats(ly.rast, ST_GeomFromWKT('""" + benton_wkt + """'), 1, 'count', true) AS loss_pixels
    FROM loss_year ly
    CROSS JOIN tree_cover tc
    CROSS JOIN forest_gain fg
""")
hansen_stats.show()

### Loss per year

In [ ]:
results = []
for yr in range(1, 24):
    row = sedona.sql(f"""
        SELECT RS_CountValue(RS_BandAsArray(ly.rast, 1), {yr}) AS pixels_lost
        FROM loss_year ly
    """).first()
    
    pixels = int(row["pixels_lost"] or 0)
    area_ha = round(pixels * 900 / 10000, 2)
    results.append({"year": 2000 + yr, "pixels_lost": pixels, "area_ha": area_ha})
    print(f"{2000 + yr}: {area_ha} ha")

In [ ]:
loss_2023 = sedona.sql("""
    SELECT pixel.geom AS geometry,
           CAST(pixel.value + 2000 AS INT) AS loss_year
    FROM (
        SELECT explode(RS_PixelAsPolygons(rast, 1)) AS pixel
        FROM loss_year
    )
    WHERE pixel.value = 23
""")
loss_2023.cache()
print(loss_2023.count())

In [ ]:
loss_2023.show(truncate=False)

In [ ]:
from sedona.spark.maps.SedonaKepler import SedonaKepler
from shapely import wkt

centroid = wkt.loads(benton_wkt).centroid
map_config = {
    "version": "v1",
    "config": {
        "mapState": {
            "latitude": centroid.y,
            "longitude": centroid.x,
            "zoom": 10
        }
    }
}

map_viz = SedonaKepler.create_map(df=loss_2023, name="Forest Loss 2023", config=map_config)
map_viz

In [ ]:
# Create a binary raster: 1 = loss in 2023, 0 = no loss
# Convert out-db → in-db → apply map algebra → serialize to GeoTIFF binary
loss_2023_raster = sedona.sql("""
    SELECT RS_AsGeoTiff(
        RS_MapAlgebra(RS_AsInDB(ly.rast), 'D', 'out = rast[0] == 23 ? 1.0 : 0.0;')
    ) AS rast
    FROM loss_year ly
""")

# Write as GeoTIFF to user storage
output_path = os.getenv("USER_S3_PATH") + "cecil/hansen_loss_2023.tiff"
loss_2023_raster.write.format("raster") \
    .option("rasterField", "rast") \
    .option("fileExtension", ".tif") \
    .mode("overwrite") \
    .save(output_path)
print(f"Exported loss 2023 raster to: {output_path}")